# Implement From Scratch — Exercises

Companion to the note [Implement From Scratch](Implement%20From%20Scratch.md).

A set of from-scratch implementation challenges that follow the note's table: classical models (linear and softmax regression, k-NN, k-means, a Gini decision tree, PCA, GMM/EM, Naive Bayes), deep-learning building blocks (a micrograd-style autograd engine, an MLP, Adam, a conv layer, causal self-attention) and the advanced rows (GCN normalisation, matrix factorisation, the VAE's KL term, Q-learning, Metropolis–Hastings). Everything is NumPy only; each result is checked against scikit-learn, SciPy, a numerical gradient or a known exact answer. PyTorch-only checks from the note are replaced by NumPy/SciPy references, and MNIST by the bundled `load_digits`.

**20 exercises** · 🧠 Concept ×2 · 💻 Code ×15 · ✍️ By hand ×3

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.datasets import load_iris, load_digits, load_breast_cancer, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.naive_bayes import MultinomialNB
from scipy.signal import correlate2d, convolve2d
from scipy.stats import norm
from scipy.integrate import quad
rng = np.random.default_rng(2024)

def _safe(f, *a, **k):
    # calls f and returns None if it is not implemented yet / raises
    try:
        return f(*a, **k)
    except Exception as e:
        print('  (call failed:', type(e).__name__, e, ')')
        return None

# shared data
X_iris, y_iris = load_iris(return_X_y=True)
X_iris = StandardScaler().fit_transform(X_iris)

## Part A · Before you start

### Exercise 1 · How do you know your implementation is right?
*🧠 Concept · ★☆☆*

The note says "implement, then compare with the library version". List **four** independent ways to test a from-scratch
implementation, and say which ones you would use for (a) a closed-form estimator, (b) a hand-written backward pass,
(c) a stochastic algorithm such as Q-learning or MCMC.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think of: library reference, numerical gradients, known analytic answers on toy inputs, invariants/properties, and convergence of the loss.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Library reference** on the same data and hyperparameters (sklearn/SciPy): best for (a).
2. **Numerical gradient check** $\frac{f(\theta+h)-f(\theta-h)}{2h}$ vs. your analytic gradient (relative error $\lesssim10^{-6}$): essential for (b).
3. **Known exact answers** on tiny inputs you can solve by hand (a 2-point regression, a 3×3 convolution, a 4-state MDP solved by value iteration): good for all three, especially (c).
4. **Invariants and properties**: probabilities sum to 1, the EM log-likelihood never decreases, attention is causal, a loss decreases on a tiny dataset you can overfit, sample moments match the target distribution: key for (c), where exact matching is impossible because of randomness.
Also: fix seeds, start with tiny shapes and `assert` shapes everywhere ([[NumPy Snippets]]).

</details>

### Exercise 2 · Why does my result differ from sklearn?
*🧠 Concept · ★★☆*

Each of these from-scratch results differs from the library. Give the most likely reason:

(a) Your logistic regression weights are larger than `LogisticRegression()`'s.
(b) Your PCA components are the library's multiplied by $-1$.
(c) Your k-means centres differ from `KMeans(n_clusters=3)` on the same data.
(d) Your decision tree picks feature 3 at the root, sklearn picks feature 2, both with the same impurity decrease.
(e) Your GMM crashes with a singular covariance matrix; sklearn's does not.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Check the default hyperparameters in the docs: `C`, `n_init`, `reg_covar`, `random_state`…

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) sklearn applies **L2 regularisation by default** (`C=1.0`, i.e. penalty $\frac{1}{2C}\lVert W\rVert^2$ on top of the summed loss).
(b) Eigenvectors/singular vectors are only defined **up to sign**; sklearn additionally flips signs deterministically (`svd_flip`). Compare `abs` values.
(c) k-means finds a **local optimum** that depends on initialisation; sklearn uses k-means++ and several restarts (`n_init`). Pass the same initial centres to compare.
(d) **Ties**: sklearn visits features in a random permutation (`random_state`) and keeps the first best split.
(e) sklearn adds `reg_covar=1e-6` to the diagonal of each covariance to keep it positive definite.

</details>

## Part B · Classical ML (rows 1–7)

### Exercise 3 · Linear regression: closed form and gradient descent
*💻 Code · ★☆☆*

Row 1 of the note. Implement `fit_closed(X, y)` (normal equations with a bias column, bias first) and `fit_gd(X, y, lr, epochs)`
(batch gradient descent on the MSE). Both should match `LinearRegression` ([[Linear Regression]]).

In [ ]:
X_reg = rng.normal(size=(200, 4))
y_reg = 0.5 + X_reg @ np.array([1.0, -2.0, 0.3, 0.0]) + 0.1 * rng.normal(size=200)

def fit_closed(X, y):
    return None

def fit_gd(X, y, lr=0.1, epochs=2000):
    return None

w_closed = fit_closed(X_reg, y_reg)
w_gd = fit_gd(X_reg, y_reg)

_lr = LinearRegression().fit(X_reg, y_reg); _ref = np.r_[_lr.intercept_, _lr.coef_]
check('closed form', w_closed, _ref)
check('gradient descent', w_gd, _ref, tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

`Xb = np.c_[np.ones(len(X)), X]`; closed form `np.linalg.solve(Xb.T @ Xb, Xb.T @ y)`; GD gradient $\frac2N X_b^\top(X_bw-y)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Use `solve`, never `inv`. GD converges because the features are standardised-ish (well-conditioned $X^\top X$).

```python
X_reg = rng.normal(size=(200, 4))
y_reg = 0.5 + X_reg @ np.array([1.0, -2.0, 0.3, 0.0]) + 0.1 * rng.normal(size=200)

def fit_closed(X, y):
    Xb = np.c_[np.ones(len(X)), X]
    return np.linalg.solve(Xb.T @ Xb, Xb.T @ y)

def fit_gd(X, y, lr=0.1, epochs=2000):
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        w -= lr * 2 / len(y) * Xb.T @ (Xb @ w - y)
    return w

w_closed = fit_closed(X_reg, y_reg)
w_gd = fit_gd(X_reg, y_reg)
```

</details>

### Exercise 4 · Softmax regression that matches sklearn exactly
*💻 Code · ★★☆*

Row 2. Implement multinomial logistic regression trained by batch gradient descent on
$L(W)=\frac1N\sum_i -\log p_{i,y_i} + \frac{\lambda}{2}\lVert W_{1:}\rVert^2$ (bias row unpenalised), where $p_i=\mathrm{softmax}(W^\top[1,x_i])$.
sklearn's `LogisticRegression(C=1.0)` minimises $\frac12\lVert W\rVert^2 + C\sum_i(-\log p_{i,y_i})$. Choose $\lambda$ so both objectives
have the same minimiser, then match sklearn's `predict_proba` on standardised iris (`X_iris, y_iris`) to 3 decimals ([[Logistic Regression]]).

In [ ]:
def softmax(Z):
    return None

def fit_softmax(X, y, lam, lr=0.5, epochs=5000):
    # returns W of shape (d+1, K), bias row first
    return None

def predict_proba_softmax(W, X):
    return None

lam_match = None
W_sm = _safe(fit_softmax, X_iris, y_iris, lam_match) if lam_match is not None else None

_ref = LogisticRegression(C=1.0, max_iter=5000, tol=1e-10).fit(X_iris, y_iris)
check('lambda that matches C=1', lam_match, 1 / len(y_iris))
check('predict_proba matches sklearn', _safe(predict_proba_softmax, W_sm, X_iris) if W_sm is not None else None, _ref.predict_proba(X_iris), tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Divide sklearn's objective by $CN$: $\frac{1}{2CN}\lVert W\rVert^2 + \frac1N\sum(-\log p)$, so $\lambda=\frac{1}{CN}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Gradient: $\frac1N X_b^\top(P-Y)$ with one-hot $Y$, plus $\lambda W$ on the non-bias rows. Subtract the row max before `exp`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\lambda = 1/(CN) = 1/150$. The softmax gradient $X_b^\top(P-Y)/N$ has the same form as for linear regression. Subtracting the row maximum
avoids overflow ([[Common Pitfalls]]: log of zero / overflow). Note: softmax is over-parameterised (adding a constant to all columns changes nothing),
so compare probabilities rather than raw weights unless both start from zero.

```python
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

def fit_softmax(X, y, lam, lr=0.5, epochs=5000):
    n, d = X.shape; K = y.max() + 1
    Xb = np.c_[np.ones(n), X]; Y = np.eye(K)[y]
    W = np.zeros((d + 1, K))
    for _ in range(epochs):
        G = Xb.T @ (softmax(Xb @ W) - Y) / n
        G[1:] += lam * W[1:]
        W -= lr * G
    return W

def predict_proba_softmax(W, X):
    return softmax(np.c_[np.ones(len(X)), X] @ W)

lam_match = 1 / len(y_iris)
W_sm = fit_softmax(X_iris, y_iris, lam_match)
```

</details>

### Exercise 5 · k-NN and k-means
*💻 Code · ★★☆*

Row 3. (a) Implement `knn_predict(Xtr, ytr, Xte, k)` with Euclidean distance and majority vote (ties to the smallest label).
(b) Implement Lloyd's algorithm `kmeans(X, init_centers, n_iter=100)` returning `(centers, labels)`; stop when the centres no longer move.
Compare with `KNeighborsClassifier` and `KMeans(init=init_centers, n_init=1)` ([[k-Nearest Neighbors]], [[Clustering]]).

In [ ]:
Xtr_i, Xte_i, ytr_i, yte_i = train_test_split(X_iris, y_iris, test_size=0.3, random_state=0)
X_blobs, _ = make_blobs(n_samples=300, centers=3, cluster_std=1.2, random_state=1)
init_c = X_blobs[[0, 1, 2]]

def knn_predict(Xtr, ytr, Xte, k=5):
    return None

def kmeans(X, init_centers, n_iter=100):
    return None, None

knn_pred = knn_predict(Xtr_i, ytr_i, Xte_i, k=5)
km_centers, km_labels = kmeans(X_blobs, init_c)

check('k-NN predictions', knn_pred, KNeighborsClassifier(5).fit(Xtr_i, ytr_i).predict(Xte_i))
_km = KMeans(n_clusters=3, init=init_c, n_init=1, tol=1e-10).fit(X_blobs)
check('k-means centres', km_centers, _km.cluster_centers_, tol=1e-6)

<details>
<summary><b>💡 Hint 1</b></summary>

Pairwise squared distances: `((A[:, None, :] - B[None, :, :])**2).sum(-1)`; neighbours via `np.argsort(D, axis=1)[:, :k]`; vote with `np.bincount(..., minlength=K).argmax()`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

k-means: assign `labels = D.argmin(1)`, update `centers[j] = X[labels == j].mean(0)`, repeat until `np.allclose(new, old)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

k-NN has no training step; all the cost is at prediction time ($O(N_{\text{train}}d)$ per query, which is why KD-trees / ANN indexes exist).
k-means alternates two steps that each decrease the inertia $\sum_i\lVert x_i-\mu_{c_i}\rVert^2$, so it converges, but only to a local optimum;
with the same initial centres it lands exactly where sklearn does.

```python
Xtr_i, Xte_i, ytr_i, yte_i = train_test_split(X_iris, y_iris, test_size=0.3, random_state=0)
X_blobs, _ = make_blobs(n_samples=300, centers=3, cluster_std=1.2, random_state=1)
init_c = X_blobs[[0, 1, 2]]

def _sqdist(A, B):
    return ((A[:, None, :] - B[None, :, :]) ** 2).sum(-1)

def knn_predict(Xtr, ytr, Xte, k=5):
    idx = np.argsort(_sqdist(Xte, Xtr), axis=1, kind='stable')[:, :k]
    K = ytr.max() + 1
    return np.array([np.bincount(ytr[row], minlength=K).argmax() for row in idx])

def kmeans(X, init_centers, n_iter=100):
    C = init_centers.astype(float).copy()
    for _ in range(n_iter):
        labels = _sqdist(X, C).argmin(1)
        newC = np.array([X[labels == j].mean(0) for j in range(len(C))])
        if np.allclose(newC, C):
            break
        C = newC
    return C, _sqdist(X, C).argmin(1)

knn_pred = knn_predict(Xtr_i, ytr_i, Xte_i, k=5)
km_centers, km_labels = kmeans(X_blobs, init_c)
```

</details>

### Exercise 6 · Gini impurity by hand
*✍️ By hand · ★☆☆*

Row 4. A node holds 6 samples of class A and 4 of class B. A candidate split sends (5 A, 1 B) left and (1 A, 3 B) right.
Compute the parent's Gini impurity $G=1-\sum_k p_k^2$, the weighted child impurity and the impurity decrease ([[Decision Trees]]).

In [ ]:
gini_parent = None
gini_children = None   # weighted by child size
gini_decrease = None

_g = lambda c: 1 - np.sum((np.array(c) / np.sum(c)) ** 2)
check('parent Gini', gini_parent, _g([6, 4]), tol=1e-3)
check('weighted child Gini', gini_children, 0.6 * _g([5, 1]) + 0.4 * _g([1, 3]), tol=1e-3)
check('impurity decrease', gini_decrease, _g([6, 4]) - 0.6 * _g([5, 1]) - 0.4 * _g([1, 3]), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Left child: $1-(5/6)^2-(1/6)^2 = 10/36$. Right child: $1-(1/4)^2-(3/4)^2=6/16$. Weights $6/10$ and $4/10$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$G_{\text{parent}} = 1-0.36-0.16 = 0.48$. Children: $0.2778$ and $0.375$; weighted $0.6\cdot0.2778+0.4\cdot0.375=0.1667+0.15=0.3167$.
Decrease $=0.48-0.3167=0.1633$. The tree picks the split with the largest decrease.

```python
gini_parent = 0.48
gini_children = 0.6 * 10/36 + 0.4 * 6/16
gini_decrease = 0.48 - gini_children
```

</details>

### Exercise 7 · A Gini decision tree
*💻 Code · ★★★*

Implement a CART classifier:

- `best_split(X, y)` returns `(feature, threshold, weighted_gini)`, trying the **midpoints** between consecutive distinct sorted values of each feature.
- `build_tree(X, y, depth, max_depth)` returns a nested dict (or a leaf class); stop when pure or at `max_depth`.
- `predict_tree(tree, X)`.

On breast cancer data, your root split must equal sklearn's depth-1 tree, and the depth-3 training accuracy must match sklearn's.

In [ ]:
X_bc, y_bc = load_breast_cancer(return_X_y=True)

def best_split(X, y):
    return None

def build_tree(X, y, depth=0, max_depth=3):
    return None

def predict_tree(tree, X):
    return None

root = best_split(X_bc, y_bc)
tree3 = build_tree(X_bc, y_bc, max_depth=3)
acc3 = None   # training accuracy of tree3

_t1 = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_bc, y_bc).tree_
check('root feature', root[0] if root is not None else None, _t1.feature[0])
check('root threshold', root[1] if root is not None else None, _t1.threshold[0], tol=1e-4)
check('depth-3 training accuracy', acc3, DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_bc, y_bc).score(X_bc, y_bc), tol=1e-9)

<details>
<summary><b>💡 Hint 1</b></summary>

Vectorise per feature: sort by the feature, one-hot the labels, `np.cumsum` gives the class counts left of every cut; the right counts are totals minus left.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Only cut where `xs[i] != xs[i+1]`; the threshold is `(xs[i] + xs[i+1]) / 2`; route `x <= threshold` to the left (sklearn's convention).

</details>

<details>
<summary><b>✅ Solution</b></summary>

With cumulative class counts each feature costs $O(N\log N)$ (the sort), so the whole tree is fast. The root split is on
feature 20 ("worst radius") just like sklearn's, and the depth-3 training accuracy matches. If sklearn ever disagrees on a node, the cause is a tie
in impurity decrease (Exercise 2(d)).

```python
X_bc, y_bc = load_breast_cancer(return_X_y=True)

def best_split(X, y):
    n = len(y); K = y.max() + 1
    best = (None, None, np.inf)
    for f in range(X.shape[1]):
        order = np.argsort(X[:, f], kind='stable')
        xs, Y = X[order, f], np.eye(K)[y[order]]
        left = np.cumsum(Y, axis=0)[:-1]            # counts left of cut i (after position i)
        right = Y.sum(0) - left
        nl = np.arange(1, n)[:, None]; nr = n - nl
        g = (nl[:, 0] * (1 - ((left / nl) ** 2).sum(1)) + nr[:, 0] * (1 - ((right / nr) ** 2).sum(1))) / n
        valid = xs[:-1] != xs[1:]
        if not valid.any():
            continue
        i = np.where(valid)[0][np.argmin(g[valid])]
        if g[i] < best[2] - 1e-12:
            best = (f, (xs[i] + xs[i + 1]) / 2, g[i])
    return best

def build_tree(X, y, depth=0, max_depth=3):
    if depth == max_depth or len(np.unique(y)) == 1:
        return int(np.bincount(y).argmax())
    f, t, _ = best_split(X, y)
    if f is None:
        return int(np.bincount(y).argmax())
    m = X[:, f] <= t
    return {'f': f, 't': t, 'l': build_tree(X[m], y[m], depth + 1, max_depth),
            'r': build_tree(X[~m], y[~m], depth + 1, max_depth)}

def predict_tree(tree, X):
    def one(x, node):
        while isinstance(node, dict):
            node = node['l'] if x[node['f']] <= node['t'] else node['r']
        return node
    return np.array([one(x, tree) for x in X])

root = best_split(X_bc, y_bc)
tree3 = build_tree(X_bc, y_bc, max_depth=3)
acc3 = np.mean(predict_tree(tree3, X_bc) == y_bc)
print(root[:2], acc3)
```

</details>

### Exercise 8 · PCA via SVD
*💻 Code · ★☆☆*

Row 5. Implement `pca_svd(X, k)` returning `(components, explained_variance_ratio, Z)`: centre $X$, take the SVD
$X_c = U\Sigma V^\top$, components are the first $k$ rows of $V^\top$, variances $\sigma_j^2/(N-1)$, scores $Z = X_cV_k$ ([[PCA]]).

In [ ]:
X_dig = load_digits().data

def pca_svd(X, k):
    return None, None, None

comps, evr, Z_pca = pca_svd(X_dig, 5)

_p = PCA(5).fit(X_dig)
check('explained variance ratio', evr, _p.explained_variance_ratio_)
check('components (up to sign)', None if comps is None else np.abs(comps), np.abs(_p.components_), tol=1e-6)
check('scores (up to sign)', None if Z_pca is None else np.abs(Z_pca), np.abs(_p.transform(X_dig)), tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

`U, S, Vt = np.linalg.svd(Xc, full_matrices=False)`; the ratio divides by the **total** variance $\sum_j\sigma_j^2$, not just the first $k$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

SVD of the centred data avoids forming the covariance $X_c^\top X_c$ (which squares the condition number). Signs are arbitrary (Exercise 2(b)),
so compare absolute values.

```python
X_dig = load_digits().data

def pca_svd(X, k):
    Xc = X - X.mean(0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S ** 2 / (len(X) - 1)
    return Vt[:k], var[:k] / var.sum(), Xc @ Vt[:k].T

comps, evr, Z_pca = pca_svd(X_dig, 5)
```

</details>

### Exercise 9 · Gaussian mixture with EM
*💻 Code · ★★★*

Row 6. Implement EM for a 1-D mixture of $K$ Gaussians: `em_gmm_1d(x, K, n_iter, seed)` returning `(pi, mu, sigma, loglik_history)`.
E-step: responsibilities $\gamma_{ik}\propto\pi_k\mathcal N(x_i\mid\mu_k,\sigma_k^2)$. M-step: weighted means, variances and weights.
Verify that the log-likelihood never decreases and that the means match `GaussianMixture` ([[Gaussian Mixture Models and EM]]).

In [ ]:
x_mix = np.r_[rng.normal(-2, 0.7, 800), rng.normal(3, 1.2, 1200)]

def em_gmm_1d(x, K=2, n_iter=200, seed=0):
    return None, None, None, None

pi_em, mu_em, sd_em, ll_hist = em_gmm_1d(x_mix)

_g = GaussianMixture(2, tol=1e-10, max_iter=2000, random_state=0).fit(x_mix[:, None])
_o = np.argsort(_g.means_.ravel())
check('means (sorted)', None if mu_em is None else np.sort(mu_em), _g.means_.ravel()[_o], tol=1e-3)
check('weights (sorted by mean)', None if mu_em is None else np.asarray(pi_em)[np.argsort(mu_em)], _g.weights_[_o], tol=1e-3)
check('log-likelihood is non-decreasing', None if ll_hist is None else bool(np.all(np.diff(ll_hist) > -1e-8)), True)

<details>
<summary><b>💡 Hint 1</b></summary>

Initialise means at random data points, $\sigma_k$ = overall std, $\pi_k=1/K$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Compute $\log\pi_k+\log\mathcal N$ for all $i,k$ and normalise with log-sum-exp; the log-likelihood is the sum over $i$ of the log-sum-exp.

</details>

<details>
<summary><b>✅ Solution</b></summary>

EM monotonically increases the likelihood because the E-step makes the lower bound tight and the M-step maximises it.
On this well-separated mixture EM recovers means $\approx(-2, 3)$, weights $\approx(0.4, 0.6)$ from any reasonable start, and matches sklearn.
In higher dimensions you must also guard against collapsing components (`reg_covar`, Exercise 2(e)).

```python
x_mix = np.r_[rng.normal(-2, 0.7, 800), rng.normal(3, 1.2, 1200)]

def em_gmm_1d(x, K=2, n_iter=200, seed=0):
    r = np.random.default_rng(seed)
    mu = r.choice(x, K, replace=False).astype(float)
    sd = np.full(K, x.std()); pi = np.full(K, 1 / K)
    hist = []
    for _ in range(n_iter):
        logp = np.log(pi) + norm.logpdf(x[:, None], mu, sd)           # (N, K)
        m = logp.max(1, keepdims=True)
        lse = m[:, 0] + np.log(np.exp(logp - m).sum(1))
        hist.append(lse.sum())
        g = np.exp(logp - lse[:, None])
        Nk = g.sum(0)
        pi = Nk / len(x)
        mu = (g * x[:, None]).sum(0) / Nk
        sd = np.sqrt((g * (x[:, None] - mu) ** 2).sum(0) / Nk)
    return pi, mu, sd, np.array(hist)

pi_em, mu_em, sd_em, ll_hist = em_gmm_1d(x_mix)
print(pi_em, mu_em, sd_em)
```

</details>

### Exercise 10 · Multinomial Naive Bayes
*💻 Code · ★★☆*

Row 7. Implement `fit_mnb(X, y, alpha)` returning `(log_prior, log_theta)` with Laplace-smoothed word probabilities
$\theta_{kj}=\frac{N_{kj}+\alpha}{N_k+\alpha d}$, and `predict_proba_mnb(model, X)` computed in log-space with log-sum-exp.
Match `MultinomialNB(alpha=1.0).predict_proba` ([[Naive Bayes]]).

In [ ]:
# synthetic bag-of-words: 3 classes, 20-word vocabulary
_theta_true = rng.dirichlet(np.ones(20) * 0.5, size=3)
y_nb = rng.integers(0, 3, 300)
X_nb = np.array([rng.multinomial(30, _theta_true[c]) for c in y_nb])

def fit_mnb(X, y, alpha=1.0):
    return None

def predict_proba_mnb(model, X):
    return None

mnb = fit_mnb(X_nb, y_nb)
proba_nb = _safe(predict_proba_mnb, mnb, X_nb) if mnb is not None else None

check('predict_proba matches MultinomialNB', proba_nb, MultinomialNB(alpha=1.0).fit(X_nb, y_nb).predict_proba(X_nb), tol=1e-8)

<details>
<summary><b>💡 Hint</b></summary>

Joint log-likelihood: `X @ log_theta.T + log_prior`; then subtract the row-wise log-sum-exp and exponentiate.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Naive Bayes is linear in log-space: $\log p(y=k\mid x)=\log\pi_k+\sum_j x_j\log\theta_{kj}-\text{const}$. Without smoothing, a single unseen word gives $\log 0=-\infty$;
without log-space, products of hundreds of small probabilities underflow.

```python
_theta_true = rng.dirichlet(np.ones(20) * 0.5, size=3)
y_nb = rng.integers(0, 3, 300)
X_nb = np.array([rng.multinomial(30, _theta_true[c]) for c in y_nb])

def fit_mnb(X, y, alpha=1.0):
    K = y.max() + 1
    counts = np.array([X[y == k].sum(0) for k in range(K)]) + alpha
    log_theta = np.log(counts / counts.sum(1, keepdims=True))
    log_prior = np.log(np.bincount(y, minlength=K) / len(y))
    return log_prior, log_theta

def predict_proba_mnb(model, X):
    log_prior, log_theta = model
    J = X @ log_theta.T + log_prior
    m = J.max(1, keepdims=True)
    J = J - (m + np.log(np.exp(J - m).sum(1, keepdims=True)))
    return np.exp(J)

mnb = fit_mnb(X_nb, y_nb)
proba_nb = predict_proba_mnb(mnb, X_nb)
```

</details>

## Part C · Deep-learning building blocks (rows 8–12)

### Exercise 11 · A micrograd-style autograd engine
*💻 Code · ★★★*

Row 8. Write a scalar `Value` class supporting `+`, `*`, `-`, unary `-`, `**` (number exponent), `/`, `tanh()`, `exp()`, also with plain
numbers on either side, and `backward()` (topological sort, then chain rule in reverse). Test it on
$f(a,b,c)=\tanh(a b + c^2)\cdot e^{a}/b - 3$ against central finite differences ([[Backpropagation]]).

In [ ]:
class Value:
    def __init__(self, data, _children=()):
        self.data, self.grad = float(data), 0.0
        self._prev, self._backward = set(_children), lambda: None
    # TODO: __add__, __radd__, __mul__, __rmul__, __neg__, __sub__, __rsub__, __pow__, __truediv__, tanh, exp, backward

def f_scalar(a, b, c):
    return (a * b + c ** 2).tanh() * a.exp() / b - 3

def autograd_grads(a0, b0, c0):
    # TODO: build Values, call backward, return [da, db, dc]
    return None

_fn = lambda a, b, c: np.tanh(a * b + c ** 2) * np.exp(a) / b - 3
_p = np.array([0.3, -1.2, 0.5]); _h = 1e-6
_num = [(_fn(*(_p + _h * e)) - _fn(*(_p - _h * e))) / (2 * _h) for e in np.eye(3)]
check('gradients match finite differences', _safe(autograd_grads, *_p), _num, tol=1e-6)

<details>
<summary><b>💡 Hint 1</b></summary>

Each op creates `out = Value(result, (self, other))` and sets `out._backward` to a closure that does `self.grad += local_grad * out.grad` (use `+=`: a node can be used twice).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`__truediv__(self, o)` can be `self * o ** -1`; `__sub__` can be `self + (-o)`. In `backward`, set `self.grad = 1` and call `_backward` on nodes in reverse topological order.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Reverse-mode autodiff = the chain rule applied in reverse topological order, accumulating gradients with `+=` because a value can feed several nodes.
This is exactly what PyTorch's autograd does on tensors.

```python
class Value:
    def __init__(self, data, _children=()):
        self.data, self.grad = float(data), 0.0
        self._prev, self._backward = set(_children), lambda: None
    @staticmethod
    def _wrap(o):
        return o if isinstance(o, Value) else Value(o)
    def __add__(self, o):
        o = Value._wrap(o); out = Value(self.data + o.data, (self, o))
        def _b():
            self.grad += out.grad; o.grad += out.grad
        out._backward = _b; return out
    def __mul__(self, o):
        o = Value._wrap(o); out = Value(self.data * o.data, (self, o))
        def _b():
            self.grad += o.data * out.grad; o.grad += self.data * out.grad
        out._backward = _b; return out
    def __pow__(self, k):
        out = Value(self.data ** k, (self,))
        def _b():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _b; return out
    def tanh(self):
        t = np.tanh(self.data); out = Value(t, (self,))
        def _b():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _b; return out
    def exp(self):
        e = np.exp(self.data); out = Value(e, (self,))
        def _b():
            self.grad += e * out.grad
        out._backward = _b; return out
    __radd__ = __add__
    __rmul__ = __mul__
    def __neg__(self): return self * -1
    def __sub__(self, o): return self + (-Value._wrap(o))
    def __rsub__(self, o): return Value._wrap(o) + (-self)
    def __truediv__(self, o): return self * Value._wrap(o) ** -1
    def backward(self):
        topo, seen = [], set()
        def build(v):
            if v not in seen:
                seen.add(v)
                for c in v._prev:
                    build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

def f_scalar(a, b, c):
    return (a * b + c ** 2).tanh() * a.exp() / b - 3

def autograd_grads(a0, b0, c0):
    a, b, c = Value(a0), Value(b0), Value(c0)
    f_scalar(a, b, c).backward()
    return [a.grad, b.grad, c.grad]
```

</details>

### Exercise 12 · An MLP in NumPy
*💻 Code · ★★☆*

Row 9 (with the bundled 8×8 `load_digits` instead of MNIST). Train a 64–64–10 MLP with ReLU and softmax cross-entropy
using mini-batch SGD and **hand-written backprop**. Reach at least 93% test accuracy ([[Neural Networks]]).

In [ ]:
_d = load_digits()
Xtr_d, Xte_d, ytr_d, yte_d = train_test_split(_d.data / 16.0, _d.target, test_size=0.25, random_state=0)

def train_mlp(X, y, hidden=64, lr=0.1, epochs=40, batch=32, seed=0):
    # returns params = (W1, b1, W2, b2)
    return None

def mlp_predict(params, X):
    return None

params_mlp = train_mlp(Xtr_d, ytr_d)
test_acc_mlp = None

if test_acc_mlp is not None:
    check('test accuracy >= 0.93', test_acc_mlp >= 0.93, True)
else:
    check('test accuracy >= 0.93', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

He initialisation: `W1 = rng.normal(0, np.sqrt(2/64), (64, hidden))`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Backward: `dZ2 = (P - Y)/B`, `dW2 = H.T @ dZ2`, `dH = dZ2 @ W2.T`, `dZ1 = dH * (Z1 > 0)`, `dW1 = X.T @ dZ1`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The backward pass is just the chain rule written in matrix form; the softmax–cross-entropy pair has the simple gradient $P-Y$.
Expect about 96–98% test accuracy after a few seconds. If accuracy is stuck near 10%, check the learning rate, the init scale and
whether you shuffle each epoch.

```python
_d = load_digits()
Xtr_d, Xte_d, ytr_d, yte_d = train_test_split(_d.data / 16.0, _d.target, test_size=0.25, random_state=0)

def train_mlp(X, y, hidden=64, lr=0.1, epochs=40, batch=32, seed=0):
    r = np.random.default_rng(seed)
    d, K = X.shape[1], 10
    W1 = r.normal(0, np.sqrt(2 / d), (d, hidden)); b1 = np.zeros(hidden)
    W2 = r.normal(0, np.sqrt(1 / hidden), (hidden, K)); b2 = np.zeros(K)
    Y = np.eye(K)[y]
    for _ in range(epochs):
        perm = r.permutation(len(X))
        for s in range(0, len(X), batch):
            i = perm[s:s + batch]; xb, yb = X[i], Y[i]
            Z1 = xb @ W1 + b1; H = np.maximum(Z1, 0)
            Z2 = H @ W2 + b2; Z2 -= Z2.max(1, keepdims=True)
            P = np.exp(Z2); P /= P.sum(1, keepdims=True)
            dZ2 = (P - yb) / len(i)
            dW2, db2 = H.T @ dZ2, dZ2.sum(0)
            dZ1 = (dZ2 @ W2.T) * (Z1 > 0)
            dW1, db1 = xb.T @ dZ1, dZ1.sum(0)
            W1 -= lr * dW1; b1 -= lr * db1; W2 -= lr * dW2; b2 -= lr * db2
    return W1, b1, W2, b2

def mlp_predict(params, X):
    W1, b1, W2, b2 = params
    return (np.maximum(X @ W1 + b1, 0) @ W2 + b2).argmax(1)

params_mlp = train_mlp(Xtr_d, ytr_d)
test_acc_mlp = np.mean(mlp_predict(params_mlp, Xte_d) == yte_d)
print(test_acc_mlp)
```

</details>

### Exercise 13 · Adam
*💻 Code · ★★☆*

Row 10. Implement `adam_step(w, g, m, v, t, lr, b1=0.9, b2=0.999, eps=1e-8)` returning `(w, m, v)` with bias correction ([[Optimizers]]).

1. Show (and check) that the **first** step from $m=v=0$ is $\approx -\text{lr}\cdot\mathrm{sign}(g)$, whatever the gradient's magnitude.
2. Minimise the ill-conditioned quadratic $f(w)=\frac12w^\top Aw-b^\top w$ with $A=\mathrm{diag}(1,100)$, $b=(1,1)$, `lr=0.01`, 5000 steps.

In [ ]:
def adam_step(w, g, m, v, t, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    return None, None, None

A_q = np.diag([1.0, 100.0]); b_q = np.array([1.0, 1.0])
g0 = np.array([3.0, -0.01, 200.0])
w_first = adam_step(np.zeros(3), g0, np.zeros(3), np.zeros(3), t=1, lr=0.1)[0]
w_adam = None   # result of 5000 Adam steps on the quadratic

check('first step = -lr * sign(g)', w_first, -0.1 * np.sign(g0), tol=1e-5)
check('Adam reaches the minimiser A^-1 b', w_adam, np.linalg.solve(A_q, b_q), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

At $t=1$: $\hat m = g$, $\hat v = g^2$, so the step is $\text{lr}\cdot g/(|g|+\epsilon)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

After bias correction, $\hat m_1 = g$ and $\hat v_1=g^2$, so $\Delta w = -\text{lr}\,g/(|g|+\epsilon)\approx-\text{lr}\,\mathrm{sign}(g)$: Adam's steps are
**scale-invariant** per coordinate. That is why it copes with the condition number 100 here, where plain GD needs $\text{lr}<2/100$ and then crawls
along the flat direction. Without bias correction the first step would be $\text{lr}\,(1-\beta_1)g/(\sqrt{1-\beta_2}\,|g|)=\frac{0.1}{0.0316}\text{lr}\approx3.2\,\text{lr}$, i.e. about three times too large.

```python
def adam_step(w, g, m, v, t, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g ** 2
    mhat, vhat = m / (1 - b1 ** t), v / (1 - b2 ** t)
    return w - lr * mhat / (np.sqrt(vhat) + eps), m, v

A_q = np.diag([1.0, 100.0]); b_q = np.array([1.0, 1.0])
g0 = np.array([3.0, -0.01, 200.0])
w_first = adam_step(np.zeros(3), g0, np.zeros(3), np.zeros(3), t=1, lr=0.1)[0]
w, m, v = np.zeros(2), np.zeros(2), np.zeros(2)
for t in range(1, 5001):
    w, m, v = adam_step(w, A_q @ w - b_q, m, v, t, lr=0.01)
w_adam = w
print(w_adam)
```

</details>

### Exercise 14 · Conv layer: forward and backward
*💻 Code · ★★★*

Row 11. For a single-channel input $X\in\mathbb R^{H\times W}$ and kernel $K\in\mathbb R^{k\times k}$, the "conv" layer of deep learning is a
**cross-correlation** with valid padding: $O_{ij}=\sum_{a,b}X_{i+a,j+b}K_{ab}$. Implement `conv2d_forward(X, K)` with loops (or strides) and
`conv2d_backward(X, K, dO)` returning `(dX, dK)`. Then express both gradients as correlations/convolutions ([[CNN]]).

In [ ]:
Xc = rng.normal(size=(7, 6)); Kc = rng.normal(size=(3, 3)); dOc = rng.normal(size=(5, 4))

def conv2d_forward(X, K):
    return None

def conv2d_backward(X, K, dO):
    return None, None

O_conv = conv2d_forward(Xc, Kc)
dX_conv, dK_conv = conv2d_backward(Xc, Kc, dOc)

check('forward = correlate2d(valid)', O_conv, correlate2d(Xc, Kc, mode='valid'))
check('dK = correlate2d(X, dO, valid)', dK_conv, correlate2d(Xc, dOc, mode='valid'))
check('dX = convolve2d(dO, K, full)', dX_conv, convolve2d(dOc, Kc, mode='full'))

<details>
<summary><b>💡 Hint 1</b></summary>

Backward by scattering: for every output position $(i,j)$, `dK += dO[i,j] * X[i:i+k, j:j+k]` and `dX[i:i+k, j:j+k] += dO[i,j] * K`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Afterwards, notice the pattern: $dK$ is the correlation of $X$ with $dO$; $dX$ is the *full* convolution (flipped kernel) of $dO$ with $K$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac{\partial L}{\partial K_{ab}}=\sum_{ij}dO_{ij}X_{i+a,j+b}$, i.e. $dK=X\star dO$ (valid correlation).
$\frac{\partial L}{\partial X_{pq}}=\sum_{ij}dO_{ij}K_{p-i,q-j}$, i.e. $dX = dO * K$ (full convolution: correlation with the 180°-rotated kernel and zero padding).
So the backward pass of a conv layer is itself a conv layer, which is how frameworks implement it.

```python
Xc = rng.normal(size=(7, 6)); Kc = rng.normal(size=(3, 3)); dOc = rng.normal(size=(5, 4))

def conv2d_forward(X, K):
    k1, k2 = K.shape
    H, W = X.shape[0] - k1 + 1, X.shape[1] - k2 + 1
    return np.array([[np.sum(X[i:i + k1, j:j + k2] * K) for j in range(W)] for i in range(H)])

def conv2d_backward(X, K, dO):
    k1, k2 = K.shape
    dX, dK = np.zeros_like(X), np.zeros_like(K)
    for i in range(dO.shape[0]):
        for j in range(dO.shape[1]):
            dK += dO[i, j] * X[i:i + k1, j:j + k2]
            dX[i:i + k1, j:j + k2] += dO[i, j] * K
    return dX, dK

O_conv = conv2d_forward(Xc, Kc)
dX_conv, dK_conv = conv2d_backward(Xc, Kc, dOc)
```

</details>

### Exercise 15 · Causal self-attention
*💻 Code · ★★☆*

Row 12. Implement single-head causal self-attention for a sequence $X\in\mathbb R^{T\times d}$:
$Q=XW_q$, $K=XW_k$, $V=XW_v$, $A=\mathrm{softmax}\big(\frac{QK^\top}{\sqrt{d_k}}+M\big)$ with $M_{ts}=-\infty$ for $s>t$, output $AV$.
Return `(out, A)`. **Vectorise** (no loop over positions). Why divide by $\sqrt{d_k}$? ([[Transformers]])

In [ ]:
T_seq, d_model, d_k = 6, 8, 4
X_seq = rng.normal(size=(T_seq, d_model))
Wq, Wk, Wv = (rng.normal(size=(d_model, d_k)) for _ in range(3))

def causal_attention(X, Wq, Wk, Wv):
    return None, None

att_out, att_A = causal_attention(X_seq, Wq, Wk, Wv)

def _loop_attention(X):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv; outs = []
    for t in range(len(X)):
        s = Q[t] @ K[:t + 1].T / np.sqrt(d_k); w = np.exp(s - s.max()); w /= w.sum()
        outs.append(w @ V[:t + 1])
    return np.array(outs)
check('matches a per-position loop', att_out, _loop_attention(X_seq))
check('attention is causal (upper triangle = 0)', None if att_A is None else np.triu(att_A, 1), np.zeros((T_seq, T_seq)))
_X2 = X_seq.copy(); _X2[-1] += 10.0     # change only the last token
_o2 = _safe(causal_attention, _X2, Wq, Wk, Wv)
check('earlier outputs ignore the future', None if att_out is None or _o2 is None or _o2[0] is None else _o2[0][:-1], None if att_out is None else att_out[:-1])

<details>
<summary><b>💡 Hint</b></summary>

Build the mask with `np.triu(np.ones((T, T)), k=1).astype(bool)` and set those scores to `-np.inf` before a row-wise stable softmax.

</details>

<details>
<summary><b>✅ Solution</b></summary>

If the entries of $q$ and $k$ have unit variance, $q^\top k$ has variance $d_k$; without scaling, large $d_k$ pushes the softmax into saturation
(one-hot rows, vanishing gradients). Dividing by $\sqrt{d_k}$ keeps the logits at unit variance. The causal mask makes position $t$ depend only on
tokens $\le t$, which is what lets a GPT be trained on all positions of a sequence in parallel.

```python
T_seq, d_model, d_k = 6, 8, 4
X_seq = rng.normal(size=(T_seq, d_model))
Wq, Wk, Wv = (rng.normal(size=(d_model, d_k)) for _ in range(3))

def causal_attention(X, Wq, Wk, Wv):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    S = Q @ K.T / np.sqrt(K.shape[1])
    S[np.triu(np.ones(S.shape), k=1).astype(bool)] = -np.inf
    S = S - S.max(1, keepdims=True)
    A = np.exp(S); A /= A.sum(1, keepdims=True)
    return A @ V, A

att_out, att_A = causal_attention(X_seq, Wq, Wk, Wv)
```

</details>

## Part D · Beyond (rows 13–17)

### Exercise 16 · GCN normalisation by hand
*✍️ By hand · ★★☆*

Row 14. A GCN layer computes $H' = \hat A H W$ with $\hat A = \tilde D^{-1/2}\tilde A\tilde D^{-1/2}$, $\tilde A = A+I$.
For the path graph $1-2-3$ and scalar features $H=(1,2,3)^\top$, $W=1$, compute $\hat A$ and $H'$ by hand ([[Graph Neural Networks]]).

In [ ]:
H_gcn = None   # [h1', h2', h3']

_A = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0.]]) + np.eye(3)
_Dm = np.diag(1 / np.sqrt(_A.sum(1)))
check("H' = Â H", H_gcn, _Dm @ _A @ _Dm @ np.array([1, 2, 3.]), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Degrees with self-loops: $(2,3,2)$. Entry $\hat A_{ij}=\tilde A_{ij}/\sqrt{\tilde d_i\tilde d_j}$, e.g. $\hat A_{12}=1/\sqrt6$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\hat A=\begin{pmatrix}1/2&1/\sqrt6&0\\1/\sqrt6&1/3&1/\sqrt6\\0&1/\sqrt6&1/2\end{pmatrix}$, so
$h_1'=0.5+2/\sqrt6=1.3165$, $h_2'=1/\sqrt6+2/3+3/\sqrt6=2.2996$, $h_3'=2/\sqrt6+1.5=2.3165$.
The symmetric normalisation keeps the scale of features stable regardless of node degree; self-loops keep each node's own features.

```python
H_gcn = [0.5 + 2/np.sqrt(6), 1/np.sqrt(6) + 2/3 + 3/np.sqrt(6), 2/np.sqrt(6) + 1.5]
```

</details>

### Exercise 17 · Matrix factorisation with ALS
*💻 Code · ★★☆*

Row 13. Ratings $R\approx PQ^\top$ with only 40% of the entries observed. Implement alternating least squares:
fixing $Q$, each user row solves a ridge problem $p_u=(Q_u^\top Q_u+\lambda I)^{-1}Q_u^\top r_u$ over the items that user rated, and vice versa.
Report the RMSE on the **held-out** entries in `als_rmse`. Then (on paper) write the BPR loss and its gradient for implicit feedback
([[Collaborative Filtering and Matrix Factorization]]).

In [ ]:
n_u, n_i, k_true = 60, 50, 3
R_true = rng.normal(size=(n_u, k_true)) @ rng.normal(size=(n_i, k_true)).T
R_obs = R_true + 0.1 * rng.normal(size=R_true.shape)
mask = rng.uniform(size=R_true.shape) < 0.4

def als(R, mask, k=3, lam=0.1, iters=30, seed=0):
    return None, None

P_als, Q_als = als(R_obs, mask)
als_rmse = None   # RMSE on ~mask entries vs R_true

_base = np.sqrt(np.mean((R_true[~mask] - R_obs[mask].mean()) ** 2))   # predict the global mean
if als_rmse is not None:
    check('ALS beats the mean baseline by far', als_rmse < 0.2 * _base, True)
else:
    check('ALS beats the mean baseline by far', None, True)

<details>
<summary><b>💡 Hint</b></summary>

For user $u$: `idx = mask[u]`, `Qu = Q[idx]`, `P[u] = np.linalg.solve(Qu.T @ Qu + lam * np.eye(k), Qu.T @ R[u, idx])`. Items: the same with the roles swapped.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each half-step is a convex ridge problem, so the objective decreases monotonically. With rank-3 data and 40% observed, held-out RMSE is at the noise level
(≈0.07, below the 0.1 noise std because the low-rank fit averages noise out), far below the global-mean baseline (≈1.7).

**BPR** (implicit feedback): for a user $u$, a consumed item $i$ and a non-consumed item $j$, maximise $\ln\sigma(x_{uij})$ with
$x_{uij}=p_u^\top(q_i-q_j)$. Gradients of the loss $-\ln\sigma(x_{uij})$: with $s=\sigma(-x_{uij})$, $\partial/\partial p_u=-s(q_i-q_j)$,
$\partial/\partial q_i=-s\,p_u$, $\partial/\partial q_j=+s\,p_u$, trained by SGD over sampled triples. VBPR (row 18) adds a visual term
$\theta_u^\top E f_i$ from image features to $x_{ui}$.

```python
n_u, n_i, k_true = 60, 50, 3
R_true = rng.normal(size=(n_u, k_true)) @ rng.normal(size=(n_i, k_true)).T
R_obs = R_true + 0.1 * rng.normal(size=R_true.shape)
mask = rng.uniform(size=R_true.shape) < 0.4

def als(R, mask, k=3, lam=0.1, iters=30, seed=0):
    r = np.random.default_rng(seed)
    P = r.normal(size=(R.shape[0], k)); Q = r.normal(size=(R.shape[1], k))
    I = lam * np.eye(k)
    for _ in range(iters):
        for u in range(R.shape[0]):
            Qu = Q[mask[u]]; P[u] = np.linalg.solve(Qu.T @ Qu + I, Qu.T @ R[u, mask[u]])
        for i in range(R.shape[1]):
            Pi = P[mask[:, i]]; Q[i] = np.linalg.solve(Pi.T @ Pi + I, Pi.T @ R[mask[:, i], i])
    return P, Q

P_als, Q_als = als(R_obs, mask)
als_rmse = np.sqrt(np.mean(((P_als @ Q_als.T) - R_true)[~mask] ** 2))
print(als_rmse)
```

</details>

### Exercise 18 · The VAE's KL term
*✍️ By hand · ★★☆*

Row 15. A VAE encoder outputs $q(z\mid x)=\mathcal N(\mu,\sigma^2)$ and the prior is $\mathcal N(0,1)$.
Derive $\mathrm{KL}(q\,\|\,p)=\frac12\big(\mu^2+\sigma^2-1-\ln\sigma^2\big)$ and evaluate it for $\mu=1$, $\sigma=0.5$.
Then explain the **reparameterisation trick** and why it is needed ([[Generative Models]]).

In [ ]:
kl_value = None

_q = norm(1, 0.5)
check('KL(N(1, 0.25) || N(0, 1))', kl_value, quad(lambda z: _q.pdf(z) * (_q.logpdf(z) - norm.logpdf(z)), -10, 10)[0], tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$\mathrm{KL}=\mathbb E_q[\ln q(z)-\ln p(z)]$; use $\mathbb E_q[(z-\mu)^2]=\sigma^2$ and $\mathbb E_q[z^2]=\mu^2+\sigma^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\ln q(z)-\ln p(z) = -\ln\sigma - \frac{(z-\mu)^2}{2\sigma^2}+\frac{z^2}{2}$. Taking $\mathbb E_q$: $-\ln\sigma-\frac12+\frac{\mu^2+\sigma^2}{2}$,
which is $\frac12(\mu^2+\sigma^2-1-\ln\sigma^2)$. For $\mu=1,\sigma=0.5$: $\frac12(1+0.25-1+\ln4)=\frac12(0.25+1.3863)=0.8181$.

**Reparameterisation**: sample $z=\mu+\sigma\epsilon$ with $\epsilon\sim\mathcal N(0,1)$. The randomness moves into $\epsilon$, so $z$ is a differentiable
function of the encoder outputs and gradients of $\mathbb E_q[\ln p(x\mid z)]$ can flow through $\mu,\sigma$ by ordinary backprop
(sampling $z\sim\mathcal N(\mu,\sigma^2)$ directly is not differentiable).

```python
kl_value = 0.5 * (1 + 0.25 - 1 - np.log(0.25))
```

</details>

### Exercise 19 · Q-learning on a frozen lake
*💻 Code · ★★★*

Row 16 (the environment is written below, no Gym needed). A deterministic 4×4 grid: start in state 0, goal 15 (reward 1),
holes {5, 7, 11, 12} (episode ends, reward 0); actions 0=left, 1=down, 2=right, 3=up; bumping into a wall leaves you in place; $\gamma=0.9$.
Implement tabular Q-learning with ε-greedy exploration and return the Q-table. The greedy value $V(s)=\max_aQ(s,a)$ must match the optimal
values from value iteration on every non-terminal state ([[Q-Learning and Policy Gradients]]).

In [ ]:
HOLES, GOAL, GAMMA = {5, 7, 11, 12}, 15, 0.9

def env_step(s, a):
    r, c = divmod(s, 4)
    if a == 0: c = max(c - 1, 0)
    elif a == 1: r = min(r + 1, 3)
    elif a == 2: c = min(c + 1, 3)
    else: r = max(r - 1, 0)
    s2 = 4 * r + c
    return s2, float(s2 == GOAL), (s2 == GOAL or s2 in HOLES)

def q_learning(episodes=3000, alpha=0.5, eps=0.3, seed=0):
    # returns Q of shape (16, 4)
    return None

Q_tab = q_learning()

def _value_iteration():
    V = np.zeros(16)
    for _ in range(200):
        for s in range(16):
            if s == GOAL or s in HOLES:
                continue
            V[s] = max(r + GAMMA * (0 if done else V[s2]) for s2, r, done in (env_step(s, a) for a in range(4)))
    return V
_nonterm = [s for s in range(16) if s != GOAL and s not in HOLES]
check('V = max_a Q matches value iteration', None if Q_tab is None else Q_tab.max(1)[_nonterm], _value_iteration()[_nonterm], tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Update: $Q(s,a)\leftarrow Q(s,a)+\alpha\big(r+\gamma\max_{a'}Q(s',a')\cdot(1-\text{done})-Q(s,a)\big)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Break ties in the greedy action randomly, cap episodes at ~100 steps, and run a few thousand episodes.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Q-learning is **off-policy**: it learns $Q^\star$ while behaving ε-greedily, because the target uses $\max_{a'}$ rather than the action actually taken.
In a deterministic environment with every state–action pair visited often, the table converges to the value-iteration solution, e.g. $V^\star(14)=1$,
$V^\star(10)=0.9$, $V^\star(0)=0.9^5\approx0.59$. DQN replaces the table with a network plus replay buffer and target network.

```python
HOLES, GOAL, GAMMA = {5, 7, 11, 12}, 15, 0.9

def env_step(s, a):
    r, c = divmod(s, 4)
    if a == 0: c = max(c - 1, 0)
    elif a == 1: r = min(r + 1, 3)
    elif a == 2: c = min(c + 1, 3)
    else: r = max(r - 1, 0)
    s2 = 4 * r + c
    return s2, float(s2 == GOAL), (s2 == GOAL or s2 in HOLES)

def q_learning(episodes=3000, alpha=0.5, eps=0.3, seed=0):
    r = np.random.default_rng(seed)
    Q = np.zeros((16, 4))
    for _ in range(episodes):
        s = 0
        for _ in range(100):
            if r.uniform() < eps:
                a = r.integers(4)
            else:
                a = r.choice(np.flatnonzero(Q[s] == Q[s].max()))
            s2, rew, done = env_step(s, a)
            Q[s, a] += alpha * (rew + GAMMA * (0 if done else Q[s2].max()) - Q[s, a])
            s = s2
            if done:
                break
    return Q

Q_tab = q_learning()
print(np.round(Q_tab.max(1).reshape(4, 4), 3))
```

</details>

### Exercise 20 · Metropolis–Hastings
*💻 Code · ★★☆*

Row 17. Sample from the unnormalised bimodal density $\tilde p(x)=0.3\,\mathcal N(x\mid-2,0.5^2)+0.7\,\mathcal N(x\mid2,1)$ with a
random-walk Metropolis sampler (Gaussian proposal, scale `step`). Implement `metropolis(log_p, x0, n, step, seed)` returning the chain and the
acceptance rate. With 50,000 steps and `step=2.5`, the sample mean and variance (after 1,000 burn-in) must match the exact values.
Compute the exact mean and variance by hand first ([[MCMC]]).

In [ ]:
def log_target(x):
    return np.log(0.3 * norm.pdf(x, -2, 0.5) + 0.7 * norm.pdf(x, 2, 1.0))

def metropolis(log_p, x0, n, step, seed=0):
    return None, None

chain, acc_rate = metropolis(log_target, 0.0, 50000, 2.5)
mh_mean = None
mh_var = None
exact_mean = None   # by hand
exact_var = None    # by hand

_m = quad(lambda x: x * np.exp(log_target(x)), -15, 15)[0]
_v = quad(lambda x: x ** 2 * np.exp(log_target(x)), -15, 15)[0] - _m ** 2
check('exact mean (by hand)', exact_mean, _m, tol=1e-6)
check('exact variance (by hand)', exact_var, _v, tol=1e-6)
check('MH mean', mh_mean, _m, tol=0.1)
check('MH variance', mh_var, _v, tol=0.25)

<details>
<summary><b>💡 Hint 1</b></summary>

Accept a proposal $x'=x+\text{step}\cdot\epsilon$ with probability $\min(1, \tilde p(x')/\tilde p(x))$: compare `np.log(u) < log_p(x') - log_p(x)`. On rejection, **repeat** the old $x$ in the chain.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Mixture moments: mean $=\sum_k\pi_k\mu_k$, $\mathbb E[x^2]=\sum_k\pi_k(\mu_k^2+\sigma_k^2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Exact: mean $=0.3(-2)+0.7(2)=0.8$; $\mathbb E[x^2]=0.3(4+0.25)+0.7(4+1)=4.775$; variance $=4.775-0.64=4.135$.
The symmetric proposal cancels in the acceptance ratio, and the normalising constant is never needed. A small `step` (e.g. 0.2) gives a high
acceptance rate but almost never jumps between the modes, so the moments come out wrong: high acceptance $\ne$ good mixing.

```python
def log_target(x):
    return np.log(0.3 * norm.pdf(x, -2, 0.5) + 0.7 * norm.pdf(x, 2, 1.0))

def metropolis(log_p, x0, n, step, seed=0):
    r = np.random.default_rng(seed)
    x, lp = x0, log_p(x0)
    out, acc = np.empty(n), 0
    for t in range(n):
        xp = x + step * r.normal()
        lpp = log_p(xp)
        if np.log(r.uniform()) < lpp - lp:
            x, lp = xp, lpp; acc += 1
        out[t] = x
    return out, acc / n

chain, acc_rate = metropolis(log_target, 0.0, 50000, 2.5)
mh_mean = chain[1000:].mean()
mh_var = chain[1000:].var()
exact_mean = 0.8
exact_var = 4.775 - 0.64
print(acc_rate, mh_mean, mh_var)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Implement From Scratch](Implement%20From%20Scratch.md).*